# Stage 3 Decontamination (Hard ID + pHash ≤ 4)

Build the **Stage 3 fine-tuning pool** and remove eval leakage before LoRA training.

**Training mix (sampler weights at train time):**

| Share | Source |
|------:|--------|
| 65% | GQA `train_balanced` |
| 10% | Visual7W pointing (`which`) |
| 10% | VQAv2 train + COCO GT boxes |
| 10% | VCR train |
| 5% | A-OKVQA train (MCQ formatted) |

**Decontamination (this notebook):**
1. **Layer 1 — Hard ID exclusions** at load time (eval splits / val questions / test2015 image ids)
2. **Layer 2 — pHash Hamming ≤ 4** against a **union** of eval reference images:
   POPE val2014 · VQAv2 test2015 · MMBench DEV/TEST · SEED-Bench · GQA testdev · A-OKVQA val

**No SSCD** — pHash only.

**Outputs** (`~/reva-data/decontamination/`):
- `stage3_exclusion_sets.json` — hard-ID exclusion sets
- `stage3_eval_reference_manifest.json` — reference image counts per benchmark
- `stage3_raw.pkl` — combined pool after hard-ID filtering
- `curriculum_vs_stage3_eval_phash.json` — pHash match log
- `phash_matches_stage3_report.csv` — audit CSV
- **`stage3_eval_clean.pkl`** — final training pool

In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


In [ ]:
!pip install imagehash pillow "numpy<2.0" tqdm matplotlib pandas pyarrow datasets huggingface_hub faiss-cpu --quiet

## 0. Manual downloads (`%%bash`)

Run the cell below once. Skips any file that already exists.

Optional large zips (off by default — flip to `1` before full pool build):

| Env var | Size (approx) |
|---------|---------------|
| `DOWNLOAD_GQA_IMAGES=1` | GQA images ~20 GB |
| `DOWNLOAD_VCR_IMAGES=1` | VCR frames ~22 GB (often blocked) |
| `DOWNLOAD_V7W_IMAGES=1` | Visual7W images ~5 GB (Meta CDN often 403) |

MMBench / SEED reference images are extracted later in Python (§5) if TSV/JSON is present.

In [ ]:
%%bash
set -euo pipefail

# --- optional large downloads (0 = skip, 1 = fetch) ---
DOWNLOAD_GQA_IMAGES=${DOWNLOAD_GQA_IMAGES:-1}
DOWNLOAD_VCR_IMAGES=${DOWNLOAD_VCR_IMAGES:-1}
DOWNLOAD_V7W_IMAGES=${DOWNLOAD_V7W_IMAGES:-1}

DATA_DIR="${REVA_REGION_DATA_ROOT:-$HOME/reva-data/region_data}"
GQA_ROOT="${REVA_GQA_ROOT:-$HOME/reva-data/gqa}"
V7W_ROOT="${REVA_VISUAL7W_ROOT:-$HOME/reva-data/visual7w}"
VQAV2_DIR="${REVA_VQAV2_ROOT:-$HOME/reva-data/vqav2}"
VCR_ROOT="${REVA_VCR_ROOT:-$HOME/reva-data/vcr}"
AOKVQA_ROOT="${REVA_AOKVQA_ROOT:-$HOME/reva-data/aokvqa}"
MMBENCH_ROOT="${REVA_MMBENCH_ROOT:-$HOME/reva-data/mmbench}"
SEED_ROOT="${REVA_SEED_ROOT:-$HOME/reva-data/seed_bench}"
HF_CACHE="${REVA_HF_CACHE_ROOT:-$HOME/reva-data/hf_cache}"

mkdir -p "$DATA_DIR/coco" "$GQA_ROOT" "$V7W_ROOT" "$VQAV2_DIR" \
         "$VCR_ROOT" "$AOKVQA_ROOT" "$MMBENCH_ROOT" "$SEED_ROOT" "$HF_CACHE"

step()     { echo "[$(date +%H:%M:%S)] $1"; }
done_msg() { echo "[$(date +%H:%M:%S)] done: $1"; }
skip_msg() { echo "[$(date +%H:%M:%S)] skip: $1"; }

# --- COCO train2014 + val2014 + instances (VQAv2 / A-OKVQA / POPE) ---
cd "$DATA_DIR/coco"
if [ ! -d train2014 ]; then
  step "COCO train2014 (~13 GB) ..."
  wget -q -c -O train2014.zip http://images.cocodataset.org/zips/train2014.zip
  unzip -qo train2014.zip && rm -f train2014.zip
  done_msg "train2014"
else skip_msg "train2014"; fi

if [ ! -d val2014 ]; then
  step "COCO val2014 (~6 GB) ..."
  wget -q -c -O val2014.zip http://images.cocodataset.org/zips/val2014.zip
  unzip -qo val2014.zip && rm -f val2014.zip
  done_msg "val2014"
else skip_msg "val2014"; fi

if [ ! -f annotations/instances_train2014.json ]; then
  step "COCO instances_train2014.json ..."
  wget -q -c -O annotations_trainval2014.zip http://images.cocodataset.org/annotations/annotations_trainval2014.zip
  unzip -qo annotations_trainval2014.zip 'annotations/instances_train2014.json'
  rm -f annotations_trainval2014.zip
  done_msg "instances_train2014.json"
else skip_msg "instances_train2014.json"; fi

# --- VQAv2 train/val + test2015 images + test question JSONs ---
cd "$VQAV2_DIR"
if [ ! -f v2_OpenEnded_mscoco_train2014_questions.json ]; then
  step "VQAv2 train questions ..."
  wget -q -c -O v2_Questions_Train_mscoco.zip https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip
  unzip -qo v2_Questions_Train_mscoco.zip && rm -f v2_Questions_Train_mscoco.zip
  done_msg "VQAv2 train questions"
else skip_msg "VQAv2 train questions"; fi

if [ ! -f v2_mscoco_train2014_annotations.json ]; then
  step "VQAv2 train annotations ..."
  wget -q -c -O v2_Annotations_Train_mscoco.zip https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip
  unzip -qo v2_Annotations_Train_mscoco.zip && rm -f v2_Annotations_Train_mscoco.zip
  done_msg "VQAv2 train annotations"
else skip_msg "VQAv2 train annotations"; fi

if [ ! -f v2_OpenEnded_mscoco_val2014_questions.json ]; then
  step "VQAv2 val questions ..."
  wget -q -c -O v2_Questions_Val_mscoco.zip https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip
  unzip -qo v2_Questions_Val_mscoco.zip && rm -f v2_Questions_Val_mscoco.zip
  done_msg "VQAv2 val questions"
else skip_msg "VQAv2 val questions"; fi

if [ ! -f v2_OpenEnded_mscoco_test-dev2015_questions.json ] || \
   [ ! -f v2_OpenEnded_mscoco_test2015_questions.json ]; then
  step "VQAv2 test-dev / test-std questions ..."
  wget -q -c -O v2_Questions_Test_mscoco.zip https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Test_mscoco.zip
  unzip -qo v2_Questions_Test_mscoco.zip && rm -f v2_Questions_Test_mscoco.zip
  done_msg "VQAv2 test questions"
else skip_msg "VQAv2 test questions"; fi

if [ ! -d test2015 ] || [ -z "$(find test2015 -maxdepth 1 -name '*.jpg' 2>/dev/null | head -1)" ]; then
  step "COCO test2015 (~12 GB) ..."
  wget -q -c -O test2015.zip http://images.cocodataset.org/zips/test2015.zip
  unzip -qo test2015.zip && rm -f test2015.zip
  done_msg "test2015"
else skip_msg "test2015"; fi

# --- GQA questions + scene graphs (+ optional images) ---
cd "$GQA_ROOT"
if [ ! -f train_balanced_questions.json ]; then
  step "GQA questions1.2 (~1.4 GB) ..."
  wget -q -c -O questions1.2.zip https://downloads.cs.stanford.edu/nlp/data/gqa/questions1.2.zip
  unzip -qo questions1.2.zip
  rm -f questions1.2.zip
  if [ -d questions1.2 ]; then
    mv questions1.2/*.json . 2>/dev/null || true
    rmdir questions1.2 2>/dev/null || true
  fi
  done_msg "GQA questions"
else skip_msg "GQA questions"; fi

if [ ! -f sceneGraphs/train_sceneGraphs.json ]; then
  step "GQA scene graphs ..."
  mkdir -p sceneGraphs
  wget -q -c -O sceneGraphs.zip https://downloads.cs.stanford.edu/nlp/data/gqa/sceneGraphs.zip
  unzip -qo sceneGraphs.zip -d sceneGraphs
  rm -f sceneGraphs.zip
  done_msg "GQA scene graphs"
else skip_msg "GQA scene graphs"; fi

if [ "$DOWNLOAD_GQA_IMAGES" = 1 ] && [ ! -d images ]; then
  step "GQA images (~20 GB) ..."
  wget -q -c -O images.zip https://downloads.cs.stanford.edu/nlp/data/gqa/images.zip
  unzip -qo images.zip && rm -f images.zip
  done_msg "GQA images"
elif [ -d images ]; then skip_msg "GQA images"; fi

# --- Visual7W pointing JSON (Stanford) + optional Meta image zip ---
cd "$V7W_ROOT"
V7W_JSON="visual7w_pointing/data/dataset.json"
if [ ! -f "$V7W_JSON" ] && [ ! -f dataset_v7w_pointing.json ]; then
  step "Visual7W pointing JSON (~98 MB, Stanford) ..."
  wget -q -c -O dataset_v7w_pointing.zip \
    http://ai.stanford.edu/~yukez/papers/resources/dataset_v7w_pointing.zip
  unzip -qo dataset_v7w_pointing.zip
  rm -f dataset_v7w_pointing.zip
  mkdir -p visual7w_pointing/data
  cp -f dataset_v7w_pointing.json "$V7W_JSON"
  done_msg "Visual7W JSON"
else skip_msg "Visual7W JSON"; fi

if [ "$DOWNLOAD_V7W_IMAGES" = 1 ]; then
  if [ ! -d visual7w_pointing/images ] || \
     [ -z "$(find visual7w_pointing/images -maxdepth 1 -name '*.jpg' 2>/dev/null | head -1)" ]; then
    step "Visual7W pointing images (Meta CDN, may 403) ..."
    if wget -q -c -O visual7w_pointing.zip \
         https://dl.fbaipublicfiles.com/visual7w/visual7w_pointing.zip && \
       unzip -qo visual7w_pointing.zip && rm -f visual7w_pointing.zip; then
      done_msg "Visual7W images"
    else
      echo "WARNING: Visual7W image zip failed (403?) — JSON-only is OK for exclusion ids"
    fi
  else skip_msg "Visual7W images"; fi
else skip_msg "Visual7W images (DOWNLOAD_V7W_IMAGES=0)"; fi

# --- VCR train/val/test JSONL (HuggingFace Rowan/vcr) ---
cd "$VCR_ROOT"
for split in train val test; do
  out="${split}.jsonl"
  if [ ! -s "$out" ]; then
    step "VCR ${split}.jsonl (HF Rowan/vcr) ..."
    wget -q -c -O "$out" \
      "https://huggingface.co/datasets/Rowan/vcr/resolve/main/original_annotations/${split}.jsonl"
    done_msg "VCR ${split}.jsonl"
  else skip_msg "VCR ${split}.jsonl"; fi
done

if [ "$DOWNLOAD_VCR_IMAGES" = 1 ]; then
  if [ ! -d vcr1images ] || [ -z "$(find vcr1images -name '*.jpg' 2>/dev/null | head -1)" ]; then
    step "VCR images (~22 GB, often blocked) ..."
    if wget -q -c -O vcr1images.zip https://dl.fbaipublicfiles.com/vcr/vcr1images.zip && \
       unzip -qo vcr1images.zip && rm -f vcr1images.zip; then
      done_msg "VCR images"
    else
      echo "WARNING: VCR images blocked — get from https://visualcommonsense.com/download/"
    fi
  else skip_msg "VCR images"; fi
else skip_msg "VCR images (DOWNLOAD_VCR_IMAGES=0)"; fi

# --- A-OKVQA train + val JSON (Allen AI S3) ---
cd "$AOKVQA_ROOT"
if [ ! -f train.json ] || [ ! -f val.json ]; then
  step "A-OKVQA annotations (Allen AI S3) ..."
  curl -fsSL https://prior-datasets.s3.us-east-2.amazonaws.com/aokvqa/aokvqa_v1p0.tar.gz \
    | tar xvz -C "$AOKVQA_ROOT"
  cp -f aokvqa_v1p0_train.json train.json
  cp -f aokvqa_v1p0_val.json val.json
  done_msg "A-OKVQA train.json + val.json"
else skip_msg "A-OKVQA JSON"; fi

# --- MMBench TSVs (pHash reference; images decoded in §5) ---
cd "$MMBENCH_ROOT"
for name in MMBench_DEV_EN MMBench_TEST_EN MMBench_DEV_EN_V11 MMBench_TEST_EN_V11; do
  if [ ! -f "${name}.tsv" ]; then
    step "MMBench ${name}.tsv ..."
    wget -q -c -O "${name}.tsv" "http://opencompass.openxlab.space/utils/VLMEval/${name}.tsv"
    done_msg "${name}.tsv"
  else skip_msg "${name}.tsv"; fi
done

# --- SEED-Bench v1 JSON (images via HF in §5 if missing) ---
cd "$SEED_ROOT"
if [ ! -f SEED-Bench.json ]; then
  step "SEED-Bench.json (HF) ..."
  wget -q -c -O SEED-Bench.json \
    "https://huggingface.co/datasets/AILab-CVC/SEED-Bench/resolve/main/SEED-Bench.json"
  done_msg "SEED-Bench.json"
else skip_msg "SEED-Bench.json"; fi

echo ""
echo "All Stage 3 downloads finished. Run §1b diagnose_stage3_paths to verify."

In [ ]:
!pip install gdown --quiet

import subprocess
from pathlib import Path

VCR_ROOT = Path(os.environ.get('REVA_VCR_ROOT') or os.path.expanduser('~/reva-data/vcr'))
VCR_ROOT.mkdir(parents=True, exist_ok=True)

SHARE_LINK = "https://drive.google.com/file/d/1T3w8TTYmeffhMlXZIIn3Tcz6K7l1p2ys/view?usp=sharing"

zip_path = VCR_ROOT / "vcr1images.zip"

if not (VCR_ROOT / "vcr1images").exists():
    import gdown
    gdown.download(url=SHARE_LINK, output=str(zip_path), quiet=False)

    print("Unzipping...")
    subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(VCR_ROOT)], check=True)
    zip_path.unlink()  # free up space
else:
    print("vcr1images/ already present, skipping")

n = sum(1 for _ in (VCR_ROOT / "vcr1images").rglob("*.jpg"))
print(f"jpg count: {n:,}")

In [ ]:
bad_lines = [l for l in result.stdout.splitlines() if "bad CRC" in l]
print(f"Corrupt entries: {len(bad_lines)}")
for l in bad_lines:
    print(l)

In [ ]:
import subprocess, json

# extract everything, tolerating the single CRC error (no check=True)
extract = subprocess.run(
    ["unzip", "-o", str(zip_path), "-d", str(VCR_ROOT)],
    capture_output=True, text=True
)

corrupt_files = [
    l.split("bad CRC")[0].split(":", 1)[-1].strip()
    for l in extract.stdout.splitlines() if "bad CRC" in l
]
print(f"Corrupt file(s): {corrupt_files}")

with open(VCR_ROOT / "corrupt_vcr_images.json", "w") as f:
    json.dump(corrupt_files, f, indent=2)

n = sum(1 for _ in (VCR_ROOT / "vcr1images").rglob("*.jpg"))
print(f"jpg count: {n:,}")

zip_path.unlink()

## 1. Setup

In [ ]:
import csv
import json
import os
import pickle
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HOME'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache')

from reva.config import ProjectionAConfig
from reva.dataset import (
    clear_phash_hash_cache,
    filter_curriculum_from_log,
    generate_curriculum_decontamination_log,
    HAMMING_THRESH,
)
from reva.stage3_dataset import (
    STAGE3_TARGET_MIX,
    apply_hard_id_exclusions,
    build_hard_exclusion_sets,
    build_stage3_raw_pool,
    collect_stage3_eval_reference_paths,
    diagnose_stage3_paths,
    summarize_pool,
    validate_stage3_sample,
    verify_stage3_modules,
)

config = ProjectionAConfig()
DECONTAM_DIR = Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination'))
DECONTAM_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = config.data_dir
GQA_ROOT = Path(os.environ.get('REVA_GQA_ROOT') or os.path.expanduser('~/reva-data/gqa'))
V7W_ROOT = Path(os.environ.get('REVA_VISUAL7W_ROOT') or os.path.expanduser('~/reva-data/visual7w'))
VQAV2_ROOT = Path(os.environ.get('REVA_VQAV2_ROOT') or os.path.expanduser('~/reva-data/vqav2'))
VCR_ROOT = Path(os.environ.get('REVA_VCR_ROOT') or os.path.expanduser('~/reva-data/vcr'))
AOKVQA_ROOT = Path(os.environ.get('REVA_AOKVQA_ROOT') or os.path.expanduser('~/reva-data/aokvqa'))
MMBENCH_ROOT = Path(os.environ.get('REVA_MMBENCH_ROOT') or os.path.expanduser('~/reva-data/mmbench'))
SEED_ROOT = Path(os.environ.get('REVA_SEED_ROOT') or os.path.expanduser('~/reva-data/seed_bench'))

COCO_TRAIN2014 = DATA_DIR / 'coco' / 'train2014'
COCO_TRAIN_INST = DATA_DIR / 'coco' / 'annotations' / 'instances_train2014.json'

# Set to a small int (e.g. 100) for dry runs; None = full datasets
MAX_SAMPLES_PER_SOURCE = None

print('Stage 3 target mix:', STAGE3_TARGET_MIX)
print(f'pHash threshold: {HAMMING_THRESH}')
print(f'MAX_SAMPLES_PER_SOURCE: {MAX_SAMPLES_PER_SOURCE}')

## 1b. Verify paths

Run after §0 downloads. Fix any `MISSING` paths before building the pool.

In [ ]:
path_report = diagnose_stage3_paths(
    gqa_root=GQA_ROOT,
    v7w_root=V7W_ROOT,
    vqav2_root=VQAV2_ROOT,
    vcr_root=VCR_ROOT,
    aokvqa_root=AOKVQA_ROOT,
    coco_train2014_dir=COCO_TRAIN2014,
    coco_train_instances=COCO_TRAIN_INST,
)

## 2. Build hard-ID exclusion sets

In [ ]:
exclusion_sets = build_hard_exclusion_sets(
    gqa_root=GQA_ROOT,
    v7w_root=V7W_ROOT,
    vqav2_root=VQAV2_ROOT,
    vcr_root=VCR_ROOT,
    aokvqa_root=AOKVQA_ROOT,
)

exclusion_export = {
    k: sorted(v) if isinstance(v, set) else v
    for k, v in exclusion_sets.items()
    if k.endswith('_ids')
}
for key in exclusion_export:
    if isinstance(exclusion_export[key], set):
        exclusion_export[key] = sorted(exclusion_export[key])

exclusion_path = DECONTAM_DIR / 'stage3_exclusion_sets.json'
serializable = {}
for key, value in exclusion_sets.items():
    if isinstance(value, set):
        serializable[key] = sorted(value)
    else:
        serializable[key] = value

with open(exclusion_path, 'w') as f:
    json.dump(serializable, f)

print(f'Saved exclusion sets -> {exclusion_path}')
for key, value in exclusion_sets.items():
    print(f'  {key}: {len(value):,}')

## 3. Build Stage 3 raw fine-tuning pool

In [ ]:
%%bash
V7W_ROOT="${REVA_VISUAL7W_ROOT:-$HOME/reva-data/visual7w}"
cd "$V7W_ROOT"

# 1) JSON (skip if you already have visual7w_pointing/data/dataset.json)
if [ ! -f visual7w_pointing/data/dataset.json ]; then
  wget -c -O dataset_v7w_pointing.zip \
    http://ai.stanford.edu/~yukez/papers/resources/dataset_v7w_pointing.zip
  unzip -qo dataset_v7w_pointing.zip
  mkdir -p visual7w_pointing/data
  cp -f dataset_v7w_pointing.json visual7w_pointing/data/dataset.json
  rm -f dataset_v7w_pointing.zip
fi

# 2) Images (~1.8 GB) — HF mirror (works when Meta CDN 403s)
if [ -z "$(find visual7w_pointing/images -maxdepth 1 -name 'v7w_*.jpg' 2>/dev/null | head -1)" ]; then
  # Option A: huggingface-cli (recommended)
  hf download lst627/COCO-Facet visual7w_images.zip \
    --repo-type dataset --local-dir .

  # Option B if huggingface-cli missing:
  # wget -c -O visual7w_images.zip \
  #   "https://huggingface.co/datasets/lst627/COCO-Facet/resolve/main/visual7w_images.zip"

  unzip -qo visual7w_images.zip
  mkdir -p visual7w_pointing/images

  # Zip unpacks to visual7w/images/*.jpg — loader expects visual7w_pointing/images/
  if [ -d visual7w/images ]; then
    cp -al visual7w/images/. visual7w_pointing/images/
  elif [ -d images ]; then
    cp -al images/. visual7w_pointing/images/
  else
    echo "Unexpected zip layout — check with: unzip -l visual7w_images.zip | head"
    exit 1
  fi

  rm -f visual7w_images.zip
fi

# 3) Sanity check
echo "jpg count: $(find visual7w_pointing/images -name 'v7w_*.jpg' | wc -l)"
ls visual7w_pointing/images/v7w_*.jpg 2>/dev/null | head -3

In [ ]:
stage3_raw = build_stage3_raw_pool(
    gqa_root=GQA_ROOT,
    v7w_root=V7W_ROOT,
    vqav2_root=VQAV2_ROOT,
    vcr_root=VCR_ROOT,
    aokvqa_root=AOKVQA_ROOT,
    coco_train2014_dir=COCO_TRAIN2014,
    coco_train_instances=COCO_TRAIN_INST,
    max_samples_per_source=MAX_SAMPLES_PER_SOURCE,
)

stage3_raw, hard_removed = apply_hard_id_exclusions(stage3_raw, exclusion_sets)
print('Hard-ID removals (post-load safety pass):', dict(hard_removed))

invalid = []
for idx, sample in enumerate(stage3_raw):
    errs = validate_stage3_sample(sample)
    if errs:
        invalid.append((idx, errs))
if invalid:
    raise RuntimeError(f'{len(invalid)} invalid samples; first errors: {invalid[:3]}')

raw_summary = summarize_pool(stage3_raw)
print('\nRaw pool summary:', json.dumps(raw_summary, indent=2))

raw_pkl = DECONTAM_DIR / 'stage3_raw.pkl'
with open(raw_pkl, 'wb') as f:
    pickle.dump(stage3_raw, f)
print(f'Saved {len(stage3_raw):,} rows -> {raw_pkl}')

## 4. Collect eval reference image union (pHash targets)

In [ ]:
ref_paths, ref_summary = collect_stage3_eval_reference_paths(
    data_dir=DATA_DIR,
    vqav2_root=VQAV2_ROOT,
    mmbench_root=MMBENCH_ROOT,
    seed_root=SEED_ROOT,
    gqa_root=GQA_ROOT,
    aokvqa_root=AOKVQA_ROOT,
    decontam_dir=DECONTAM_DIR,
    include_seed_video_frames=False,
)

assert len(ref_paths) > 0, 'No eval reference images found — run downloads / prior decontamination notebooks first.'

ref_manifest_path = DECONTAM_DIR / 'stage3_eval_reference_manifest.json'
with open(ref_manifest_path, 'w') as f:
    json.dump(ref_summary, f, indent=2)
print(f'Saved reference manifest -> {ref_manifest_path}')

## 5. Clear pHash cache & run decontamination (pHash only)

In [ ]:
clear_phash_hash_cache(Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')))

LOG_PATH = DECONTAM_DIR / 'curriculum_vs_stage3_eval_phash.json'
log_path = generate_curriculum_decontamination_log(
    curriculum_samples=stage3_raw,
    all_ref_paths=ref_paths,
    config=config,
    output_json_path=str(LOG_PATH),
    type='phash',
)
print(f'pHash log saved: {log_path}')

## 6. Summarise pHash matches

In [ ]:
with open(log_path) as f:
    log_data = json.load(f)

phash_matches = log_data['phash_matches']
unique_removed = set(m['train_idx'] for m in phash_matches)

print(f'pHash match rows: {len(phash_matches):,}')
print(f'Unique rows flagged: {len(unique_removed):,} / {len(stage3_raw):,}')
print(f'Corrupt/unreadable: {len(log_data.get("corrupt_indices", [])):,}')
print(f'Remaining after pHash: {len(stage3_raw) - len(unique_removed):,}')

removed_by_source = Counter(
    stage3_raw[m['train_idx']]['source'] for m in phash_matches
)
print('\nRemoved by source:')
for src, n in removed_by_source.most_common():
    print(f'  {src:10s}: {n:,}')

hamming_dist = Counter(m['hamming_distance'] for m in phash_matches)
if hamming_dist:
    plt.figure(figsize=(6, 3))
    xs = sorted(hamming_dist)
    ys = [hamming_dist[x] for x in xs]
    plt.bar(xs, ys)
    plt.xlabel('Hamming distance')
    plt.ylabel('Matches')
    plt.title('Stage 3 pHash matches')
    plt.show()

## 7. Export clean pool + CSV audit

In [ ]:
stage3_clean = filter_curriculum_from_log(stage3_raw, str(log_path))

csv_path = DECONTAM_DIR / 'phash_matches_stage3_report.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            'train_idx', 'source', 'question_id', 'image_id',
            'train_image', 'reference_image', 'hamming_distance',
        ],
    )
    writer.writeheader()
    for m in phash_matches:
        idx = m['train_idx']
        row = stage3_raw[idx]
        writer.writerow({
            'train_idx': idx,
            'source': row.get('source', ''),
            'question_id': row.get('question_id', ''),
            'image_id': row.get('image_id', ''),
            'train_image': m['train_image'],
            'reference_image': m['reference_image'],
            'hamming_distance': m['hamming_distance'],
        })
print(f'Saved {len(phash_matches):,} match rows -> {csv_path}')

clean_pkl = DECONTAM_DIR / 'stage3_eval_clean.pkl'
with open(clean_pkl, 'wb') as f:
    pickle.dump(stage3_clean, f)

clean_summary = summarize_pool(stage3_clean)
manifest = {
    'target_mix': STAGE3_TARGET_MIX,
    'hamming_threshold': HAMMING_THRESH,
    'raw_summary': raw_summary,
    'clean_summary': clean_summary,
    'reference_counts': {k: v for k, v in ref_summary.items() if k != 'seed_stats'},
    'hard_removed': dict(hard_removed),
    'phash_removed': len(unique_removed),
}
manifest_path = DECONTAM_DIR / 'stage3_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'\nFinal clean pool: {clean_pkl}')
print(json.dumps(clean_summary, indent=2))

## 8. Verification

Runs synthetic checks on `stage3_dataset.py` (hard-ID filter + pHash pipeline) and validates the exported clean pool schema.

In [ ]:
verify_stage3_modules()

assert clean_pkl.exists(), f'Missing {clean_pkl}'
with open(clean_pkl, 'rb') as f:
    loaded = pickle.load(f)

assert len(loaded) == len(stage3_clean)
assert len(loaded) <= len(stage3_raw)

schema_errors = []
for i, sample in enumerate(loaded[: min(500, len(loaded))]):
    schema_errors.extend(f'row {i}: {e}' for e in validate_stage3_sample(sample))
assert not schema_errors, schema_errors[:5]

overlap_qids = set()
for sample in loaded:
    src = sample['source']
    qid = str(sample['question_id'])
    if src == 'gqa' and qid in exclusion_sets['gqa_question_ids']:
        overlap_qids.add(('gqa', qid))
    if src == 'aokvqa' and qid in exclusion_sets['aokvqa_question_ids']:
        overlap_qids.add(('aokvqa', qid))
    if src == 'vqav2' and qid.isdigit() and int(qid) in exclusion_sets['vqav2_question_ids']:
        overlap_qids.add(('vqav2', qid))
assert not overlap_qids, f'Hard-ID leakage in clean pool: {list(overlap_qids)[:5]}'

removed_indices = set(m['train_idx'] for m in phash_matches)
corrupt_indices = set(log_data.get('corrupt_indices', []))
clean_keys = {(s['image_path'], s['question_id']) for s in loaded}
for idx in removed_indices | corrupt_indices:
    s = stage3_raw[idx]
    assert (s['image_path'], s['question_id']) not in clean_keys
assert len(loaded) == len(stage3_raw) - len(removed_indices | corrupt_indices)

print('Verification passed.')
print(f'  clean rows: {len(loaded):,}')
print(f'  sources: {Counter(s["source"] for s in loaded)}')
print(f'  manifest: {manifest_path}')

In [ ]:
# Verifying if name resolution is applied on VCR
import pickle
import re
from pathlib import Path

pkl = Path(os.environ.get("REVA_STAGE3_CLEAN_PKL") or os.path.expanduser("~/reva-data/decontamination/stage3_eval_clean.pkl"))
with open(pkl, "rb") as f:
    pool = pickle.load(f)

vcr = [s for s in pool if s["source"] == "vcr"]
print(f"VCR rows: {len(vcr):,}")

# Show a few examples
for s in vcr[:5]:
    print("Q:", s["question"])
    print("A:", s["answer"])
    print("---")